# D2Mamba Research — Notebook B: Branches 3 & 4 (`exp/3-haar-phase-fast` + `exp/4-phaseformer-physics-mamba`)

**Kaggle Settings**:
- Accelerator: **GPU T4 x2** (or GPU T4 / P100)
- Internet: **ON**
- Datasets attached: **UIEB** (`raw-890`/`reference-890`) + **LSUI** (`input`/`GT`)
- Protocol: **From scratch, 200 epochs, seed=42, official zero-leakage test splits** (auto `RepConv.fuse()` at test & profiling).

In [ ]:
REPO = "https://github.com/asserelzeki6/D2Mamba-Research.git"

!rm -rf /kaggle/working/exp3 /kaggle/working/exp4
!git clone --depth 1 -b exp/3-haar-phase-fast {REPO} /kaggle/working/exp3
!git clone --depth 1 -b exp/4-phaseformer-physics-mamba {REPO} /kaggle/working/exp4

!cd /kaggle/working/exp3 && python3 verify_branch.py --device cuda
!cd /kaggle/working/exp4 && python3 verify_branch.py --device cuda

In [ ]:
# Branch 3 (Haar-Phase-Fast + RepConv + H/8 Physics Mamba) — UIEB (200 epochs from scratch)
!cd /kaggle/working/exp3 && python3 train_kaggle.py \
    --dataset uieb \
    --scan_mode haar_phase_fast \
    --epochs 200 \
    --batch_size 8 \
    --lr 3e-4 \
    --img_size 256 \
    --amp \
    --lambda_phys 0.1 \
    --seed 42 \
    --save_dir /kaggle/working/checkpoints/exp3_uieb

In [ ]:
# Branch 3 (Haar-Phase-Fast + RepConv + H/8 Physics Mamba) — LSUI (200 epochs from scratch)
!cd /kaggle/working/exp3 && python3 train_kaggle.py \
    --dataset lsui \
    --scan_mode haar_phase_fast \
    --epochs 200 \
    --batch_size 8 \
    --lr 3e-4 \
    --img_size 256 \
    --amp \
    --lambda_phys 0.1 \
    --seed 42 \
    --save_dir /kaggle/working/checkpoints/exp3_lsui

In [ ]:
# Branch 4 (Phaseformer Fourier Phase-Amplitude Cross-Gating + Haar U-Net + Physics Mamba) — UIEB (200 epochs)
!cd /kaggle/working/exp4 && python3 train_kaggle.py \
    --dataset uieb \
    --scan_mode phaseformer_physics_mamba \
    --epochs 200 \
    --batch_size 8 \
    --lr 3e-4 \
    --img_size 256 \
    --amp \
    --lambda_phys 0.1 \
    --seed 42 \
    --save_dir /kaggle/working/checkpoints/exp4_uieb

In [ ]:
# Branch 4 (Phaseformer Fourier Phase-Amplitude Cross-Gating + Haar U-Net + Physics Mamba) — LSUI (200 epochs)
!cd /kaggle/working/exp4 && python3 train_kaggle.py \
    --dataset lsui \
    --scan_mode phaseformer_physics_mamba \
    --epochs 200 \
    --batch_size 8 \
    --lr 3e-4 \
    --img_size 256 \
    --amp \
    --lambda_phys 0.1 \
    --seed 42 \
    --save_dir /kaggle/working/checkpoints/exp4_lsui

In [ ]:
import json
from pathlib import Path

for run in ["exp3_uieb", "exp3_lsui", "exp4_uieb", "exp4_lsui"]:
    p = Path("/kaggle/working/checkpoints") / run / "eval_summary.json"
    if not p.is_file():
        print(f"{run}: NOT FOUND")
        continue
    d = json.loads(p.read_text())
    m = d["test_metrics"]
    hw = d["hardware_profile"]
    p256 = hw["256x256"]
    p720 = hw["1280x720"]
    print(
        f"{run:10s} | PSNR(RGB/Y)={m['psnr_rgb']:.2f}/{m['psnr_y']:.2f} dB | "
        f"SSIM(RGB/Y)={m['ssim_rgb']:.4f}/{m['ssim_y']:.4f} | "
        f"Params={hw['total_params_M']:.3f}M | GFLOPs={p256['gflops']} | "
        f"256={p256['fps']:.1f} FPS ({p256['latency_ms']:.1f} ms) | "
        f"720p={p720['fps']:.1f} FPS | VRAM={p256['peak_vram_mb']:.0f} MB | "
        f"JIT={hw['torch_jit_trace']['passed']}"
    )